<a href="https://colab.research.google.com/github/nyrxe/obj_file_classification/blob/collab/vox.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install trimesh numpy scipy
import trimesh, numpy as np, os, glob
print("trimesh", trimesh.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 735.5/735.5 kB 12.0 MB/s eta 0:00:00
trimesh 4.8.3


In [3]:
import os, glob

DATA_DIR = "/content"   # this is where uploaded files land
print("Files in /content:", os.listdir(DATA_DIR))

ply_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.ply")))
assert ply_files, "No .ply found — make sure you uploaded it!"
print("Found .ply files:", ply_files)


Files in /content: ['.config', '.ipynb_checkpoints', 'example.ply', 'sample_data']
Found .ply files: ['/content/example.ply']


In [4]:
import trimesh

PLY_PATH = "/content/example.ply"
mesh = trimesh.load(PLY_PATH, process=False)  # keep your face colors as-is

print("verts:", len(mesh.vertices))
print("faces:", len(mesh.faces))
print("is_watertight:", mesh.is_watertight)
print("extents (xyz size):", mesh.extents)

# Check per-face RGBA existence
has_face_colors = (
    hasattr(mesh, "visual") and
    getattr(mesh.visual, "face_colors", None) is not None and
    len(mesh.visual.face_colors) == len(mesh.faces)
)
print("face colors per face:", has_face_colors)
if has_face_colors:
    print("face_colors dtype/shape:", mesh.visual.face_colors.dtype, mesh.visual.face_colors.shape)


verts: 2048
faces: 4120
is_watertight: True
extents (xyz size): [0.04 0.04 0.01]
face colors per face: True
face_colors dtype/shape: uint8 (4120, 4)


In [5]:
TARGET_RES = 128  # try 256 later if you want more detail
longest = float(max(mesh.extents))
pitch = longest / TARGET_RES
print(f"target_res={TARGET_RES}, pitch={pitch:.8f} (mesh units)")


target_res=128, pitch=0.00031250 (mesh units)


In [6]:
vg_surface = mesh.voxelized(pitch=pitch)
vg_filled = vg_surface.fill()

print("surface grid shape:", vg_surface.shape, "pitch:", vg_surface.pitch)
print("filled  grid shape:", vg_filled.shape)
print("surface voxels:", len(vg_surface.sparse_indices))
print("filled  voxels:", len(vg_filled.sparse_indices))


surface grid shape: (129, 129, 33) pitch: [0.0003125 0.0003125 0.0003125]
filled  grid shape: (129, 129, 33)
surface voxels: 303757
filled  voxels: 303757


In [7]:
import os, numpy as np
out_dir = "/content/voxel_out"
os.makedirs(out_dir, exist_ok=True)

# 5a) Save as box-meshes (handy for quick preview in any mesh viewer)
vox_mesh_surface = vg_surface.as_boxes()
vox_mesh_filled  = vg_filled.as_boxes()
surf_ply  = os.path.join(out_dir, "vox_surface_boxes.ply")
fill_ply  = os.path.join(out_dir, "vox_filled_boxes.ply")
vox_mesh_surface.export(surf_ply)
vox_mesh_filled.export(fill_ply)
print("saved:", surf_ply, "and", fill_ply)

# 5b) Save sparse occupancy with metadata
np.savez_compressed(
    os.path.join(out_dir, "vox_occupancy_sparse.npz"),
    indices=vg_filled.sparse_indices,                 # (N, 3) int voxel coords (i,j,k)
    pitch=np.array([vg_filled.pitch], np.float32),    # scalar voxel size
    transform=vg_filled.transform.astype(np.float32)  # 4x4: index -> world coords
)
print("saved:", os.path.join(out_dir, "vox_occupancy_sparse.npz"))


saved: /content/voxel_out/vox_surface_boxes.ply and /content/voxel_out/vox_filled_boxes.ply
saved: /content/voxel_out/vox_occupancy_sparse.npz


In [8]:
import numpy as np
from trimesh.proximity import ProximityQuery

# Use the filled voxel grid from earlier steps
vg = vg_filled

# Occupied voxel indices (i, j, k)
idx = vg.sparse_indices
N = len(idx)
print(f"Filled voxels: {N}")

# Convert voxel indices -> world coordinates (voxel centers)
homo = np.hstack([idx, np.ones((N, 1), dtype=idx.dtype)])  # (N,4)
centers_world = (vg.transform @ homo.T).T[:, :3]            # (N,3)
print("centers_world shape:", centers_world.shape)


Filled voxels: 303757
centers_world shape: (303757, 3)


In [10]:
!apt-get -qq install -y libspatialindex-dev
!pip install -q rtree


Selecting previously unselected package libspatialindex6:amd64.
(Reading database ... 126666 files and directories currently installed.)
Preparing to unpack .../libspatialindex6_1.9.3-2_amd64.deb ...
Unpacking libspatialindex6:amd64 (1.9.3-2) ...
Selecting previously unselected package libspatialindex-c6:amd64.
Preparing to unpack .../libspatialindex-c6_1.9.3-2_amd64.deb ...
Unpacking libspatialindex-c6:amd64 (1.9.3-2) ...
Selecting previously unselected package libspatialindex-dev:amd64.
Preparing to unpack .../libspatialindex-dev_1.9.3-2_amd64.deb ...
Unpacking libspatialindex-dev:amd64 (1.9.3-2) ...
Setting up libspatialindex6:amd64 (1.9.3-2) ...
Setting up libspatialindex-c6:amd64 (1.9.3-2) ...
Setting up libspatialindex-dev:amd64 (1.9.3-2) ...
Processing triggers for libc-bin (2.35-0ubuntu3.8) ...
/sbin/ldconfig.real: /usr/local/lib/libumf.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbb.so.12 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur

In [11]:
# Build proximity structure on your original mesh
pq = ProximityQuery(mesh)

# For very large N, do it in chunks to be memory-safe
chunk = 200_000
face_ids = np.empty(N, dtype=np.int64)
for s in range(0, N, chunk):
    e = min(s + chunk, N)
    # on_surface returns (points, distances, face_indices)
    _, _, fidx = pq.on_surface(centers_world[s:e])
    face_ids[s:e] = fidx

# Map face -> RGBA
assert (hasattr(mesh, "visual") and getattr(mesh.visual, "face_colors", None) is not None), "Mesh has no per-face colors."
face_rgba = mesh.visual.face_colors  # (F,4) uint8
voxel_rgba = face_rgba[face_ids]     # (N,4) uint8
print("voxel_rgba shape/dtype:", voxel_rgba.shape, voxel_rgba.dtype)


voxel_rgba shape/dtype: (303757, 4) uint8


In [12]:
import os, trimesh
out_dir = "/content/voxel_out"
os.makedirs(out_dir, exist_ok=True)

pc = trimesh.points.PointCloud(vertices=centers_world, colors=voxel_rgba)
colored_points_ply = os.path.join(out_dir, "voxels_filled_colored_points.ply")
pc.export(colored_points_ply)
print("saved:", colored_points_ply)


saved: /content/voxel_out/voxels_filled_colored_points.ply


In [13]:
colored_boxes_ply = os.path.join(out_dir, "voxels_filled_colored_boxes.ply")

boxes = vg.as_boxes()
try:
    # Try setting per-face colors uniformly per cube by expanding RGBA
    # Each cube contributes 12 triangles (36 verts) → we broadcast colors
    # Safer approach: set face colors after creation
    F = len(boxes.faces)
    # Build per-face colors by assigning same RGBA to each face of a cube.
    # Each cube has 12 faces; faces are grouped per cube in as_boxes output.
    if F % 12 != 0:
        raise RuntimeError("Unexpected face count grouping; cannot assign per-cube colors safely.")
    faces_per_cube = 12
    C = F // faces_per_cube
    if C != len(voxel_rgba):
        # In some versions, as_boxes can produce a different cube ordering.
        # If counts mismatch, skip coloring to avoid wrong mapping.
        raise RuntimeError("Cube count doesn't match voxel count; skipping color assignment.")

    # Repeat each voxel color 12 times (one per face of the cube)
    face_colors = np.repeat(voxel_rgba[:C], faces_per_cube, axis=0)
    boxes.visual.face_colors = face_colors
    boxes.export(colored_boxes_ply)
    print("saved:", colored_boxes_ply)
except Exception as e:
    print("Could not assign colors to boxes reliably:", e)
    # Save uncolored boxes as a fallback
    fallback_ply = os.path.join(out_dir, "voxels_filled_boxes_uncolored.ply")
    boxes.export(fallback_ply)
    print("saved:", fallback_ply, "(uncolored)")


saved: /content/voxel_out/voxels_filled_colored_boxes.ply
